# Is the endogenous actor's latent an editable OBJECT HANDLE? (grabbability, §4)

**Direction:** `endogenous-action-interactive-world` · GRU · 2026-07-29. Companion to
`endogenous_actor_observer.ipynb`, which established the *identifiability* half (the goal-directed L3 actor's
latent is far more linearly readable about (pos, vel) than its observer twin). This notebook asks the harder
question: is that latent **grabbable** — can a *foreign write mechanism* (latent surgery, i.e. NOT the trained
action channel) pick up 'object 0' and move it, cleanly and selectively?

**This run rebuilds the test after a review that found three problems in v1** (all fixed here):
1. the waterfall injected the true target row into *every* column and hid each editor's own step-0 decode;
2. rollouts used no-op actions, which is *off-policy* for an actor that always acts;
3. predictor quality was never measured — a blurry model fails editing for the wrong reason.

So we (a) added a **quality gate**, (b) trained **stronger predictors** (deeper encoder/decoder, 512 hidden,
5-step free-run objective, 25k iterations) and compare them against the weak ones, (c) widened the editor
line-up, and (d) added an **action-channel** control.

*Source:* `runs/endogenous/{L3 (weak), L3s0, L3s1 (strong)}`; `scripts/eval_editability_endogenous.py` →
`runs/endogenous/editability_metrics_v2.json`; waterfalls in `runs/endogenous/edit_figs_v2/`.

> **⚠ Known deviation from the repo standard.** These runs used **`obs_noise_std = 0.05`**; every prior dataset
> (0–8, including dataset 4 behind the exogenous-action work) uses **0.2**. It leaked in from a `scripts/play.py`
> display default — not a deliberate choice. All endogenous runs share it, so comparisons *within* this thread are
> valid, but **absolute RMSE / R² are not comparable to the earlier notebooks** (less noise ⇒ easier prediction and
> probing; note the noise floor here is 0.066 rather than ≈0.2). Training default restored to 0.2; re-run queued.

## Run definitions (copied from the canonical registry `ENDOGENOUS_RUNS.md`)

**Roles.** *actor* = its policy head emits the action applied to the world (trained on prediction **plus**
REINFORCE into the shared GRU trunk). *observer* = identical architecture, same observations, **fed the actor's
actions**, trained on prediction only, never acts.

**Suffix key:** no suffix / `b` = the original ("**weak**") configuration differing only in seed; `s` = the
"**strong**" configuration; trailing digit = seed. **"weak" vs "strong" means exactly two configurations:**

| | weak | strong |
|---|---|---|
| hidden units | 256 | **512** |
| encoder / decoder | single Linear each | **2-layer MLP encoder + residual MLP decoder** |
| rollout objective | next-step only | next-step **+ 5-step free-run (multistep)** |
| training iterations | 6 000 | **25 000** |

| code | descriptive label (used in figures) | level / world | goal |
|---|---|---|---|
| `L3` | L3 force+goal · weak 256h · seed 0 | `force` dynamics, lethal (collision + wall) | survive |
| `L3s0` | L3 force+goal · **STRONG** 512h · seed 0 | same | survive |
| `L3s1` | L3 force+goal · **STRONG** 512h · seed 1 | same | survive |

The strong configuration was introduced on 2026-07-29 specifically to test whether the editability negative was an
artifact of a weak predictor.

In [ ]:
# [1] Setup — load the §4 metrics.
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
OK = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73','grey':'#8a8f98','red':'#D55E00','sky':'#56B4E9','pink':'#CC79A7'}
def style_ax(ax):
    for s in ['top','right']: ax.spines[s].set_visible(False)
    ax.grid(alpha=0.25, lw=0.6)
M = json.load(open('../../../../runs/endogenous/editability_metrics_v2.json'))
RUNS = [r for r in ['L3','L3s0','L3s1'] if r in M]
LABEL = {'L3':'weak\n256h, 6k it','L3s0':'strong seed0\n512h+MLP, 25k it','L3s1':'strong seed1\n512h+MLP, 25k it'}
print('runs:', RUNS)

## Definitions

| term | formula / meaning | units | better |
|---|---|---|---|
| **quality: free-run RMSE** | open-loop rollout from the edit frame under the TRUE actions vs the TRUE future obs | obs [0,1] | ↓ |
| **copy-previous-frame baseline** | `RMSE(obs[t], obs[t−1])` on the eval trace — what you get by predicting no change | obs [0,1] | reference |
| **observation noise floor** | `RMSE(noisy obs, clean render)` — the irreducible RMSE | obs [0,1] | reference (lower bound) |
| **step RMSE to target** | `RMSE(rollout step s, static post-edit target render)` | obs [0,1] | ↓ |
| **reach (% of swap)** | obs change at object-0's *target* rays, as % of what the true-state swap does there | % | → 100 |
| **collateral (% of swap)** | same change measured at the *other object's* rays | % | ↓ |
| **ghost ratio** | mean intensity at object-0's *vacated* (pre-edit) rays, edited ÷ unsteered. **1.0 = the object never left**; 0 = fully gone | ratio | ↓ |
| **selectivity** | reach / (reach + collateral) | frac | ↑ |
| **persistence** | obs-change at steps 10–14 ÷ at step 0 (does the edit stick) | ratio | ↑ |

**Editors** (all write to the passive latent, then free-run): *Readout injection* (linear pos-probe pseudo-inverse),
*Global-PCA projection* (inject ↔ project onto the global PCA hull), *PCA geodesic* (inject ↔ project onto a local
tangent, 50 iters), *MLP-probe gradient* (steer h until a frozen MLP probe reads the target). **References:**
*True-swap* (teacher-force the true post-edit obs — a soft reference, belief-inertia-limited) and *Decoder gradient*
(an **oracle**: gradient descent on h against the true target obs — proves a state rendering the target exists).

**Rollout modes:** `self` = the model's own policy acts on its imagined world (in-distribution for the actor);
`noop` = no action (the passive convention of the earlier object-individuation work).

In [ ]:
# [2] Fig 1 — predictor quality, with dataset baselines as dashed reference lines.
#     (a) next-step RMSE; (b) RMSE against the post-edit target vs rollout step, per editor.
BL = M[RUNS[0]]['baselines']
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))
ax = axes[0]; x = np.arange(len(RUNS)); w = 0.38
for j, role in enumerate(['actor','observer']):
    ax.bar(x + (j-0.5)*w, [M[r][role]['quality']['nextstep_rmse'] for r in RUNS], w,
           label=role, color=[OK['blue'],OK['orange']][j])
for val, lab, c in [(BL['identity_rmse'], f"copy-previous-frame baseline ({BL['identity_rmse']:.3f})", OK['grey']),
                    (BL['noise_floor_rmse'], f"observation noise floor ({BL['noise_floor_rmse']:.3f})", OK['green'])]:
    ax.axhline(val, color=c, ls='--', lw=1.2, label=lab)
ax.set_xticks(x); ax.set_xticklabels([LABEL[r] for r in RUNS], fontsize=8)
ax.set_title('(a) next-step prediction RMSE  (↓ better)', fontsize=10); ax.set_ylabel('obs RMSE', fontsize=9)
style_ax(ax); ax.legend(fontsize=7.5)
ax = axes[1]
MODE0 = 'self'
for ed, c in zip(['Unsteered-ref' ,'Readout injection','PCA geodesic','MLP-probe gradient','Decoder gradient','True-swap'],
                 [OK['grey'],OK['sky'],OK['green'],OK['blue'],OK['pink'],'k']):
    if ed == 'Unsteered-ref': continue
    curve = M[RUNS[-1]]['actor'][MODE0][ed]['step_rmse_to_target']
    ax.plot(range(len(curve)), curve, lw=1.8, marker='o', ms=3, label=ed, color=c)
ax.set_xlabel('rollout step after the edit'); ax.set_ylabel('RMSE vs post-edit target render', fontsize=9)
ax.set_title(f'(b) does the edit LAND and HOLD?  [{LABEL[RUNS[-1]]}]', fontsize=10)
style_ax(ax); ax.legend(fontsize=7.5)
fig.suptitle('Fig 1 — predictor quality (with baselines) and per-step edit fidelity', y=1.03, fontsize=11)
fig.tight_layout(); display(fig); plt.close(fig)

In [ ]:
# [3] Fig 2 — the object-handle scorecard: GHOST is the decisive axis (1.0 = the object never left its old spot).
MODE = 'self'
EDS = ['Readout injection','Global-PCA projection','PCA geodesic','MLP-probe gradient','Decoder gradient','True-swap']
COLS = [OK['grey'],OK['sky'],OK['green'],OK['blue'],OK['pink'],'k']
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))
panels = [(axes[0], 'ghost', '(a) ghost ratio — did object 0 LEAVE its old location?', 'ghost ratio (↓)'),
          (axes[1], 'reach', '(b) reach (% of a true swap)', 'reach % (→100)')]
for ax, metric, ttl, ylab in panels:
    xr = np.arange(len(RUNS)); bw = 0.13
    for j, ed in enumerate(EDS):
        ax.bar(xr + (j-2.5)*bw, [M[r]['actor'][MODE][ed][metric] for r in RUNS], bw, label=ed, color=COLS[j])
    ax.set_xticks(xr); ax.set_xticklabels([LABEL[r] for r in RUNS], fontsize=8)
    ax.set_title(ttl, fontsize=10); ax.set_ylabel(ylab, fontsize=9); style_ax(ax)
axes[0].axhline(1.0, color=OK['red'], ls='--', lw=1.2)
axes[0].text(0.02, 1.02, 'object never leaves', color=OK['red'], fontsize=8, transform=axes[0].get_yaxis_transform())
axes[1].legend(fontsize=7, ncol=2)
fig.suptitle(f'Fig 2 — object-handle scorecard on the ACTOR (rollout mode: {MODE})', y=1.02, fontsize=11)
fig.tight_layout(); display(fig); plt.close(fig)

## The action-channel control — what exactly is being measured

This is the complement to latent surgery: instead of writing into `h`, can object 0 be driven to the target through
the **action channel the model was actually trained on**? The procedure is:

1. **In the REAL simulator**, start from the edit-frame world state and run a bang-bang **PD controller** on object 0
   (`action = sign((target − position) − 4·velocity)`, object 1 held at no-op) for 15 steps. This closes **93–95%**
   of the distance, which establishes that the action channel genuinely has the authority to do the job.
2. **Record that exact action sequence**, then replay it inside the model's **imagination**: warm the model on the
   real observations up to the edit frame, then roll out 15 steps **autoregressively** (feeding its own predictions
   back) with those actions supplied.

**How are the actions "input" if the model generates its own?** The policy head is simply **bypassed**. The action
enters through the decoder conditioning — `decode_action(h, a)` — which is the same pathway used during training;
we substitute the controller's `a` for the one the policy would have sampled. This works identically for the actor
and the observer (both share the architecture), and **both are reported** — the figure shows the actor; the observer
is in the table below and behaves the same.

**Why is imagined performance so poor when the animations show good predictions?** Two reasons, and panel (c)
separates them:

- **The animations are teacher-forced** — the model sees the real observation every step, so they only ever show a
  **one-step-ahead** prediction. Here the rollout is **closed-loop for 15 steps** and error compounds. Panel (c)
  shows exactly this: step-1 error ≈ **0.12** (right in line with the teacher-forced next-step RMSE, i.e. the
  animations are not lying), growing to ≈ **0.35** by step 15.
- **The controller's actions are off-policy** — the trained actor would never choose them, so this also taxes
  off-policy generalisation. The two effects are confounded here, which is why the earlier "button, not a handle"
  phrasing was retracted: we cannot cleanly attribute the failure to the affordance not living in the state.

See the closed-loop animations in `endogenous_agent_animations.ipynb` for the same phenomenon shown visually.

In [ ]:
# [4] Fig 3 — ACTION CHANNEL vs LATENT SURGERY, and where the imagined error comes from.
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
xr = np.arange(len(RUNS)); bw = 0.35
ax = axes[0]
ax.bar(xr - bw/2, [100*M[r]['actor']['action_interface']['real_frac_distance_closed'] for r in RUNS], bw,
       label='REAL simulator (channel authority)', color=OK['green'])
ax.bar(xr + bw/2, [M[r]['actor']['action_interface']['imagined_reach'] for r in RUNS], bw,
       label="model's IMAGINATION (same actions)", color=OK['blue'])
ax.set_xticks(xr); ax.set_xticklabels([LABEL[r] for r in RUNS], fontsize=8)
ax.set_title('(a) can the ACTION channel move object 0 to the target?', fontsize=10)
ax.set_ylabel('% distance closed  /  reach %', fontsize=9); style_ax(ax); ax.legend(fontsize=7.5)
ax = axes[1]
ax.bar(xr - bw/2, [M[r]['actor']['action_interface']['imagined_ghost'] for r in RUNS], bw,
       label='action channel (imagined)', color=OK['blue'])
ax.bar(xr + bw/2, [M[r]['actor'][MODE]['PCA geodesic']['ghost'] for r in RUNS], bw,
       label='latent surgery (PCA geodesic)', color=OK['orange'])
ax.axhline(1.0, color=OK['red'], ls='--', lw=1.2, label='1.0 = object never left')
ax.set_xticks(xr); ax.set_xticklabels([LABEL[r] for r in RUNS], fontsize=8)
ax.set_title('(b) ghost ratio: action channel vs latent surgery', fontsize=10)
ax.set_ylabel('ghost ratio (↓)', fontsize=9); style_ax(ax); ax.legend(fontsize=7.5)
ax = axes[2]
for r, c in zip(RUNS, [OK['orange'], OK['blue'], OK['sky']]):
    cur = M[r]['actor']['action_interface'].get('model_vs_real_step_rmse')
    if cur: ax.plot(range(1, len(cur)+1), cur, lw=1.8, marker='o', ms=3, label=LABEL[r], color=c)
ax.axhline(M[RUNS[0]]['baselines']['noise_floor_rmse'], color=OK['green'], ls='--', lw=1.2,
           label='observation noise floor')
ax.set_xlabel('closed-loop rollout step'); ax.set_ylabel('model-vs-REAL obs RMSE', fontsize=9)
ax.set_title('(c) WHY: one-step is fine, free-run error compounds', fontsize=10)
style_ax(ax); ax.legend(fontsize=7.5)
fig.suptitle('Fig 3 — the action-channel control, and the compounding-error explanation', y=1.03, fontsize=11)
fig.tight_layout(); display(fig); plt.close(fig)

In [ ]:
# [5] Full table — every editor, both rollout modes, actor vs observer.
rows = ['| run | role | editor | reach % | collat % | ghost | select | persist |','|---|---|---|---|---|---|---|---|']
for r in RUNS:
    for role in ['actor','observer']:
        for ed in EDS:
            c = M[r][role][MODE][ed]
            rows.append(f"| {r} | {role} | {ed} | {c['reach']:.1f} | {c['collat']:.1f} | "
                        f"{c['ghost']:.3f} | {c['select']:.2f} | {c['persist']:.2f} |")
display(Markdown('\n'.join(rows)))
q = ['| run | role | free-run RMSE ↓ | sharpness TV ratio →1 | next-step RMSE ↓ | action: real closes % | action: imagined ghost |','|---|---|---|---|---|---|---|']
for r in RUNS:
    for role in ['actor','observer']:
        Q = M[r][role]['quality']; A = M[r][role]['action_interface']
        q.append(f"| {r} | {role} | {Q['freerun_rmse']:.4f} | {Q['sharpness_tv_ratio']:.3f} | "
                 f"{Q['nextstep_rmse']:.4f} | {100*A['real_frac_distance_closed']:.0f}% | {A['imagined_ghost']:.3f} |")
display(Markdown('\n'.join(q)))

In [ ]:
# [6] Waterfalls — each column is that column's OWN free-run from the edit frame (no teacher forcing).
for r in RUNS:
    for role in ['actor','observer']:
        display(Markdown(f'**{LABEL[r]} — {role}** (rollout mode `self`)'))
        display(Image(filename=f'../../../../runs/endogenous/edit_figs_v2/edit_{r}_{role}_self.png'))

In [ ]:
# [7] Fig 4 — REVISION of the identifiability headline: the actor-vs-observer gap at weak vs strong capacity.
ID = json.load(open('../../../../runs/endogenous/eval_metrics.json'))
groups = [('weak L3 (256h, 6k it)', ['L3','L3b']), ('strong L3 (512h+MLP, 25k it)', ['L3s0','L3s1']),
          ('strong L2 — no goal (control)', ['L2s0'])]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (mk, ttl) in zip(axes, [('pos_r2_lin','(a) position R² linear'), ('vel_r2_lin','(b) velocity R² linear'),
                                ('fiber_mlp','(c) fiber residual MLP (lower = more canonical)')]):
    labels, vals = [], []
    for gname, runs_ in groups:
        d = [ID[r]['actor'][mk] - ID[r]['observer'][mk] for r in runs_ if r in ID]
        labels.append(gname); vals.append(np.mean(d) if d else np.nan)
    cols = [OK['orange'], OK['blue'], OK['grey']]
    ax.bar(range(len(vals)), vals, color=cols)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=7.5, rotation=12)
    ax.set_title(ttl, fontsize=10); ax.set_ylabel('Δ (actor − observer)', fontsize=9); style_ax(ax)
fig.suptitle('Fig 4 — the identifiability advantage of acting SHRINKS when both models are trained to strength',
             y=1.04, fontsize=11)
fig.tight_layout(); display(fig); plt.close(fig)

## Two methodological notes

### Why the ground-truth column is a *static* target render

The `GT (sim target)` column repeats the post-edit target render for every row rather than showing an evolving
simulated future. This is deliberate, and it is a **limitation to keep in mind when reading the per-step curve**.

A truly matched ground truth would mean: apply the edit to the *real simulator*, then step it forward using the
actions the model actually chooses. But **each editor changes the latent, and the latent determines the policy**, so
every editor would induce a *different* action sequence and therefore a *different* ground-truth future — and the
unsteered rollout would legitimately diverge too, because its belief no longer matches the edited world. There is no
single common reference. Freezing the target render gives **one shared, editor-independent reference** for all
columns, at the cost of being the right answer only at step 0 and drifting from any real future thereafter.

**Consequence for the metrics:** step-0 quantities (reach, collateral, ghost — the ones the verdict rests on) are
unaffected, since at step 0 the static render *is* the correct target. The per-step RMSE curve should be read as
*"how long does the edit keep resembling the intended post-edit scene"*, **not** as prediction error against a true
future. (This is the same frozen-target caveat flagged in `METRICS_AND_EDITORS.md`.)

### How much was the prediction-vs-survival loss balance tuned? **Not at all.**

The actor's objective is a fixed weighted sum — `predictor_loss + 1.0·policy_loss + 0.5·value_loss +
0.01·entropy_bonus` — and **those weights are the first values tried; no sweep was run.** The prediction-vs-control
balance is therefore an arbitrary, unvalidated hyperparameter, and since the actor-vs-observer contrast is by
construction a function of it, a **weight sweep is the most obvious missing control**. It is cheap (≈90 min/point on
this GPU) and is the natural next experiment.

**On making the penalty natural instead of weighted** (the "death = unpredictability" idea): the appeal is real —
an arbitrary λ between two unrelated objectives is unsatisfying, whereas if death were *intrinsically* costly to a
predictor there would be one objective. The substrate for it already exists here (death → 4 frames of pure noise →
rebirth at a random state, i.e. a maximally unpredictable event). **But it does not by itself remove the need for
RL:** the prediction loss shapes the *representation* through backprop, while the action head can only be improved
by a signal that says *which action was better*, and that requires either differentiating through the world (our
simulator is not differentiable) or a policy-gradient estimator. **The clean way to get what you want is to keep
REINFORCE as the mechanism but make the reward BE the prediction error** — i.e. `reward = −(prediction error)`,
which is exactly the SMiRL / free-energy formulation: the agent then acts to keep its world predictable, and dying
is punished automatically because death is the least predictable thing that can happen. That converts the arbitrary
λ into a single self-consistent objective while remaining trainable. **This is a concrete, cheap next experiment**
and I would run it before any further weight sweeps.

## Summary — interpretation (calibrated)

**Current results (updated 2026-07-29):** **The latent is NOT an editable object handle — and this is now established at model strength, not inferred from a blurry model.** Across two independently-seeded strong actors (512 hidden, MLP encoder/decoder, 5-step free-run objective, 25k iterations) and both rollout modes, every *structural* editor is inert: ghost ratio **0.998–1.010** (1.0 = the object never leaves its old location) with reach 0.3–6%. **The decisive control:** on the *same models, same decoder, same rollout*, the **decoder-gradient oracle** (ghost **0.004–0.012**, reach 89–93%) and the **true-state swap** (ghost ≈ 0, reach 100%) both succeed completely. If blur or weak prediction caused the failure, these would fail too. So a state that renders the target exists and the model can roll it out — probe-directed writes simply cannot reach it. The failure is in the **edit map's reachability**, not the predictor. Notably the structural editors became *more* inert as the predictor improved (PCA geodesic reach 28% → 4%).

**Revision to the identifiability headline.** With both models trained to strength the actor-vs-observer gap **shrinks sharply**: position R² Δ +0.173/+0.135 (weak) → **+0.030/+0.005** (strong), because the observer catches up (0.589 → 0.863). The velocity gap survives but ~3× smaller (+0.17 → **+0.044/+0.060**), and the canonicality gap **flips sign** — the strong actor is now consistently *more* canonical (fiber MLP Δ **−0.070/−0.084**, vs +0.060/+0.071 weak). The **goal-specificity holds**: the strong no-goal L2 control is a null (Δpos +0.018, Δvel −0.015). Reading: goal-directed agency mainly **accelerates** the emergence of linearly-readable structure and yields a modest, durable gain in velocity readability and canonicality — not the large representational advantage the weak-model run suggested.

**Caveats.** (1) The stronger models did **not** fix the blur — free-run sharpness only 0.607 → 0.633 TV ratio and free-run RMSE slightly worse; capacity + a multistep objective were not enough, so rollout quality remains a real limitation of this architecture. (2) The **action-channel control is not a clean 'button' result**: a PD controller in the *real simulator* closes 93–95% of the distance to the target, but the model's *imagination* of those same actions barely moves the object (reach 2–6%, ghost ≈ 0.98). Those actions are **off-policy**, so this conflates 'the action channel does not transfer' with 'off-policy generalization is poor'. The honest summary is that the model's imagined world supports **no** tested intervention route — latent or action — except direct decoder optimization and fresh observational evidence. It is an **on-policy predictor, not an intervention-supporting simulator**. (3) GRU only; god's-hand; 2 objects; N=64 edits; 2 seeds at L3.
